# Import tools
Simulation, Data Manipulation and Math Operations

In [1]:
# Importing libraries & frameworks 
import random 
import genesis as gs
import numpy as np
import cv2
import json
from pathlib import Path

[I 08/20/25 21:38:18.108 2718360] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


# Initializing Genesis

Below is defined some setup the scene and camera as well as the definition of the Rigid Body Entities used in the Dataset Generation (Bottle). 


In [2]:
# Genesis initialization 
gs.init(
    backend=gs.cuda,
    seed=None,
    precision='32',
    debug=False,
    eps=1e-12,
    logging_level=None,
    theme='dark',
    logger_verbose_time=False
)

# Creating the scene
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, 0, 1.0),
        camera_lookat=(0, 0, 0.15),
        camera_fov=30,
        max_FPS=600
    ),
    vis_options=gs.options.VisOptions(show_world_frame=False),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=False, 
    show_FPS=False
)

# Plane (ground) 
plane = scene.add_entity(gs.morphs.Plane())
mouse_pos = [0, 0.0, 0.015] 
mouse = scene.add_entity(
    gs.morphs.Mesh(
        file="../computer_mouse/mouse.stl", 
        pos=mouse_pos,            
        euler=(0, 0, 0),
        collision=True,
        visualization=True,
        convexify = True,
        parse_glb_with_trimesh = True,
        merge_submeshes_for_collision = True,
        fixed=True,          
        scale=0.05                      
    )
)


# Camera Setup
cam = scene.add_camera(
    pos    =(3, -1.5, 0.2),
    lookat = mouse_pos,
    res    = (640, 480),
    fov    = 30,
    GUI    = False,
    up = (0, 0, 1),

)
# Building the Scene
scene.build()

[Genesis] [21:38:22] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [21:38:22] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [21:38:22] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [21:38:22] [INFO] Consider setting 'performance_mode=True' in production to maximise runtime speed, if significantly increasing compilation time is not a concern.
[Genesis] [21:38:22] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.63 GB.
[Genesis] [21:38:22] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [21:38:22] [WARNING] Scene.show_FPS is deprecated. Please use Scene.profiling_options.show_FPS
[Genesis] [21:38:23] [INFO] Scene <a2dff2b> created.
[Genesis] [21:38:23] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <cfafc35>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [21:38:23] [INFO] Adding <gs.RigidEntity>. id

## Pre-dataset Parameters

Below some parameters for the dataset creation are defined.

In [ ]:
# Output directory
dataset_dir = Path("./mouse/mouse_dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

# Verification directory
image_dir = Path("./mouse/mouse_verification_dataset")
image_dir.mkdir(parents=True, exist_ok=True)

# Set the density of the grid
n_theta = 16  # elevation steps -> from top to equator
n_phi = 32   # azimuthal steps -> around the object
radius = 0.5 # Camera distance from the ccentral point

# hemispherical coordinates
theta = np.linspace(0, np.pi / 2, n_theta) # from 0 to 90deg
phi = np.linspace(0, np.pi, n_phi) # from 0 to 180deg

# Create a 2D grid of theta and phi values
phi_grid, theta_grid = np.meshgrid(phi, theta)

# Spherical -> Cartesian Coordinates
x = radius * np.sin(theta_grid) * np.cos(phi_grid)
y = radius * np.sin(theta_grid) * np.sin(phi_grid)
z = radius * np.cos(theta_grid)

# Stack into (x, y, z) points
grid_points = np.stack([x.ravel(), y.ravel(), z.ravel()], axis=1) 

print(f"Grid points: {grid_points.shape}")  # (n_views, 3)


Grid points: (512, 3)


## Image extraction Loop

In this loop all the randomization and data save happens.

In [6]:
from tqdm.auto import tqdm
for i, (x, y, z) in enumerate(tqdm(grid_points, desc="Captured images:")):
    # Randomize camera lookat
    rand_lookat_x = random.uniform(-0.03, 0.03)
    rand_lookat_y = random.uniform(-0.03, 0.03)
    rand_lookat_z = random.uniform(0.02, 0.11)   
    
    cam_lookat = [rand_lookat_x, rand_lookat_y, rand_lookat_z]
    cam_pos = [x, y, z]
    
    if  x == 0.0 and y == 0.0:
        up = [0, 1, 0]
    else: 
        up = [0, 0, 1]
    # Change camera position
    cam.set_pose(
        pos= cam_pos,
        lookat= cam_lookat
    )
    
    #  View setup 
    view_id = f"view_{i}"  #
    view_dir = dataset_dir / view_id
    img_view_dir = image_dir / view_id
    view_dir.mkdir(parents=True, exist_ok=True)
    img_view_dir.mkdir(parents=True, exist_ok=True)

    # Get RGB-D data
    for x in range(5):
        scene.step()
        render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False)

    rgb, depth, seg, normal_map = render_output
    bottle_pixels = np.where(np.array(seg) == 1)
    bottle_pixels = np.transpose(bottle_pixels)  # Shape: (num_pixels, 2)

    # Save data
    np.save(view_dir / "rgb.npy", rgb)
    np.save(view_dir / "depth.npy", depth)
    np.save(view_dir / "segmentation.npy", seg)
    np.save(view_dir / "normal.npy", normal_map)

    # Save PNGs for visual verification
    cv2.imwrite(str(img_view_dir / "rgb.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

    # Store metadata
    data_metadata = {
        "view_id": view_id,
        "camera_pos": cam_pos,
        "camera_lookat": cam_lookat,
        "up": up
    }

    # Save metadata
    with open(view_dir / "metadata.json", "w") as f:
        json.dump(data_metadata, f, indent=4)

    for _ in range(200):
        scene.step()

Captured images:: 100%|██████████| 512/512 [03:06<00:00,  2.74it/s]
